<a href="https://colab.research.google.com/github/binhvd/AI-Vision/blob/master/01_Introduction_Fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction & Fundamentals

This lab walks through the whole pipeline from Lectures 1 and 2, in the same order:

| Part | Lecture topic |
|---|---|
| 1 | Digital image, coordinate system, bit depth |
| 2 | Sampling & quantization (Genesis of the digital signal) |
| 3 | Image types & color models (RGB / HSV) |
| 4 | Noise, point processing, histograms |
| 5 | Spatial filtering & convolution |
| 6 | Edge detection |
| 7 | Segmentation & classification |
| 8 | Final challenge: a full *image in → information out* pipeline |

**How to work with this notebook**

* There are **10 tasks**, marked `TODO 1` … `TODO 10`. Replace the `None` placeholders.
* Most tasks end with an `assert` or a plot that tells you whether you got it right.
* Cells **will raise errors until you fill in the TODOs** — that is expected.
* Cells marked *worked example* are already complete: read them and run them, they set
  up variables the tasks need.
* Only run cells top to bottom; later parts reuse variables from earlier ones.

## Part 0: Setup

Run this cell first. It imports everything and defines a small `show()` helper so you
can spend your time on the image operations instead of on matplotlib boilerplate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from skimage import data, color, filters, exposure, feature, util, measure, morphology
from scipy import ndimage as ndi

np.random.seed(0)
plt.rcParams["image.cmap"] = "gray"
plt.rcParams["figure.dpi"] = 110


def show(*images, titles=None, cmap="gray", vmin=None, vmax=None, size=3.2):
    # Display one or more images side by side.
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(size * n, size + 0.4))
    axes = np.atleast_1d(axes)
    for i, (ax, im) in enumerate(zip(axes, images)):
        ax.imshow(im, cmap=None if (im.ndim == 3) else cmap, vmin=vmin, vmax=vmax)
        if titles is not None:
            ax.set_title(titles[i], fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


print("Setup OK — numpy", np.__version__)

## Part 1: The digital image as a matrix

> *An image is a function f(x, y). A digital image is that function with x, y and f
> discrete and finite — i.e. a 2D matrix.*

We use the classic `camera` test image (8-bit grayscale).

In [ ]:
cam = data.camera()
show(cam, titles=["data.camera()"], size=4)

### Describe the image numerically

Everything the lecture calls a property of the image is readable straight off the array.
`bit_depth` is the number of bits per pixel `k`, and `n_gray_levels` is `L = 2**k` —
both follow from the dtype (see the *Intensity and Bit Depth* slide).

*(Worked example — read the code and run it.)*

In [ ]:
height, width = cam.shape
n_pixels = cam.size
bit_depth = cam.dtype.itemsize * 8
n_gray_levels = 2 ** bit_depth

print(f"size           : {height} rows x {width} columns")
print(f"pixels         : {n_pixels}")
print(f"dtype          : {cam.dtype}")
print(f"bit depth k    : {bit_depth}")
print(f"gray levels L  : {n_gray_levels}")
print(f"min / max      : {cam.min()} / {cam.max()}")
print(f"mean intensity : {cam.mean():.1f}")

assert n_pixels == height * width
assert n_gray_levels == 256

### TODO 1: Indexing and the image coordinate system

Remember the convention from the lecture: **the origin (0,0) is the top-left corner,
`x` runs *down* over rows and `y` runs *right* over columns** — so NumPy indexing is
`img[x, y]`, which is the opposite order from the usual maths plotting convention.

1. Read the intensity of the pixel at row 100, column 200.
2. On a **copy**, paint a white square over rows 40–70 and columns 30–60.
3. Crop the cameraman's head: roughly rows 30–130, columns 190–290.

In [ ]:
# TODO 1a: intensity at row 100, column 200
value = None
print("f(100, 200) =", value)

# TODO 1b: paint a white (255) square on a COPY, rows 40..70, cols 30..60
marked = cam.copy()
# ... your code here

# TODO 1c: crop rows 30..130 and columns 190..290
head = None

show(marked, head, titles=["white square patch", f"crop {head.shape}"])
assert head.shape == (100, 100)
assert marked[50, 50] == 255 and cam[50, 50] != 255  # original untouched

## Part 2: Sampling and quantization

The two steps that turn a continuous physical signal into a digital array:

* **Sampling** → digitizing the coordinates `(x, y)` → *spatial resolution* (M x N).
* **Quantization** → digitizing the amplitude `f` → *intensity resolution* (L = 2^k).

Here we go the other way: we start from a good image and *destroy* resolution, to see
what each step costs us.

### Undersampling and aliasing

`downsample` keeps every k-th row and column — simple sub-sampling with **no**
pre-filtering, which is exactly what produces aliasing. Watch the fine brick texture
break down as `k` grows: the pattern does not just get coarser, it turns into a
*different*, wrong pattern.

*(Worked example — read the code and run it.)*

In [ ]:
def downsample(img, k):
    return img[::k, ::k]


brick = data.brick()
versions = [downsample(brick, k) for k in (1, 2, 4, 8)]
show(*versions, titles=[f"k={k}  {v.shape}" for k, v in zip((1, 2, 4, 8), versions)])

assert downsample(brick, 4).shape == (brick.shape[0] // 4, brick.shape[1] // 4)

### TODO 2: Coarse quantization and false contouring

Write `quantize(img, k)` that re-quantizes an 8-bit image to `k` bits, i.e. to
`L = 2**k` gray levels, while keeping the output in the 0–255 display range.

*Hint:* one clean way is to keep only the `k` most significant bits.
Bit shifting does that in one line: `(img >> (8 - k)) << (8 - k)`.

In [ ]:
def quantize(img, k):
    # TODO 2: reduce img to 2**k gray levels, still stored as uint8 in 0..255
    return None


levels = [1, 2, 3, 4, 8]
qs = [quantize(cam, k) for k in levels]
show(*qs, titles=[f"k={k} bit  (L={2**k})" for k in levels], vmin=0, vmax=255)

for k in levels:
    assert len(np.unique(quantize(cam, k))) <= 2 ** k, f"too many levels for k={k}"

**Think about it:** at which `k` do you first notice *false contouring* (smooth
gradients breaking into visible bands)? Look at the sky, not at the coat.

## Part 3: Image types and color models

Binary (1 bit) → grayscale (8 bit) → color (24 bit = 3 stacked grayscale channels).

### A color image is a stack of grayscale images

The three channels are sliced out along the last axis and shown **as grayscale images**,
then stacked back together. Look at the flag and the red mission patch: bright in R,
dark in G and B. An N x M x 3 color image really is three stacked scalar images.

*(Worked example — read the code and run it.)*

In [ ]:
astro = data.astronaut()
print("shape N x M x 3 :", astro.shape)

R = astro[:, :, 0]
G = astro[:, :, 1]
B = astro[:, :, 2]

rebuilt = np.stack([R, G, B], axis=2)

show(astro, R, G, B, titles=["RGB", "R channel", "G channel", "B channel"])
assert np.array_equal(rebuilt, astro)

### RGB → HSV

`color.rgb2hsv` gives the three components; the hue is then rotated by 0.5 (half a turn
around the color wheel, wrapping in [0,1)) and converted back with `color.hsv2rgb`.

Note how H and S carry *color* while V carries *brightness* — V on its own looks like an
ordinary grayscale photo. That separation is the point of the next task.

*(Worked example — read the code and run it.)*

In [ ]:
hsv = color.rgb2hsv(astro)
H, S, V = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]

hsv_shifted = hsv.copy()
hsv_shifted[:, :, 0] = (hsv[:, :, 0] + 0.5) % 1.0
shifted_rgb = color.hsv2rgb(hsv_shifted)

show(H, S, V, shifted_rgb, titles=["Hue", "Saturation", "Value", "Hue + 0.5"])
assert np.all((hsv_shifted[:, :, 0] >= 0) & (hsv_shifted[:, :, 0] < 1))

### TODO 3: Why the lecture says HSI/HSV "decouples color from brightness"

We take a washed-out, low-contrast color image and boost its contrast **two ways**:

* **(a) naive:** run histogram equalization on R, G and B *independently*;
* **(b) correct:** convert to HSV, equalize **only the V channel**, convert back.

Use `exposure.equalize_hist`. Both give more contrast — but compare the *colors*.

In [ ]:
chel = util.img_as_float(data.chelsea())
faded = 0.35 * chel + 0.42          # squash the dynamic range -> low contrast

# TODO 3a: equalize each RGB channel independently, restack to an N x M x 3 image
eq_rgb = None

# TODO 3b: convert to HSV, equalize ONLY channel 2 (V), convert back to RGB
hsv_f = None
eq_v = None

show(faded, eq_rgb, eq_v,
     titles=["low contrast input", "(a) equalize R,G,B separately", "(b) equalize V only"],
     size=3.6)

**Question:** which version keeps the cat's fur the right color, and why does the
other one shift the hues? Write one sentence in the cell below.

*Your answer:*



## Part 4: Noise, point processing, histograms

A **point operation** computes `s = T(r)` for each pixel independently of its neighbors.

### Add noise

Two degraded copies of `cam` (as float in [0,1]) that we will keep using:

* Gaussian noise, variance 0.01 — models sensor heat / electronic gain;
* salt-and-pepper noise, 8% of pixels — models dead pixels / switching errors.

`rng=0` fixes the seed so your numbers match the ones in the solution.

*(Worked example — read the code and run it.)*

In [ ]:
cam_f = util.img_as_float(cam)

noisy_gauss = util.random_noise(cam_f, mode="gaussian", var=0.01, rng=0)
noisy_sp = util.random_noise(cam_f, mode="s&p", amount=0.08, rng=0)

show(cam_f, noisy_gauss, noisy_sp,
     titles=["clean", "gaussian noise", "salt & pepper"], vmin=0, vmax=1)

### The three classic intensity transformations

All three are one-liners on **8-bit** input (`L = 256`):

| transform | formula |
|---|---|
| negative | `s = L - 1 - r` |
| log | `s = c * log(1 + r)` with `c = 255 / log(1 + r_max)` |
| power-law (gamma) | `s = c * r**gamma`, scaled so the output fills 0..255 |

They are applied to `data.moon()`, a dark and low-contrast image. Read the code and
match each function against its formula above.

*(Worked example — read the code and run it.)*

In [ ]:
def negative(img):
    return (255 - img.astype(np.int16)).astype(np.uint8)


def log_transform(img):
    r = img.astype(np.float64)
    c = 255.0 / np.log(1 + r.max())
    return np.clip(c * np.log(1 + r), 0, 255).astype(np.uint8)


def gamma_transform(img, gamma):
    r = img.astype(np.float64) / 255.0
    return np.clip(255.0 * (r ** gamma), 0, 255).astype(np.uint8)


moon = data.moon()
show(moon, negative(moon), log_transform(moon),
     gamma_transform(moon, 0.4), gamma_transform(moon, 2.5),
     titles=["original", "negative", "log", "gamma=0.4", "gamma=2.5"],
     vmin=0, vmax=255, size=2.8)

assert negative(np.array([[0, 255]], np.uint8)).tolist() == [[255, 0]]
assert gamma_transform(moon, 1.0).max() <= 255

**Think about it:** `gamma=0.4` and `gamma=2.5` are inverses of each other. Which one
brightens the dark craters, and which one is closer to what a display does to your
signal (the non-linearity that gamma correction compensates)?

### Read a histogram

`plot_histogram` draws the 256-bin intensity histogram of an 8-bit image;
`np.bincount(img.ravel(), minlength=256)` gives the counts in one call. You will reuse
this helper in three later cells.

Note the shape for `moon`: a *narrow cluster* means low contrast, a *broad spread* means
high contrast.

*(Worked example — read the code and run it.)*

In [ ]:
def plot_histogram(img, ax, title=""):
    counts = np.bincount(img.ravel(), minlength=256)
    ax.bar(np.arange(256), counts, width=1.0)
    ax.set_xlim(0, 255)
    ax.set_xlabel("intensity level (0-255)")
    ax.set_ylabel("pixel count")
    ax.set_title(title, fontsize=9)


fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].imshow(moon, vmin=0, vmax=255)
axes[0].axis("off")
axes[0].set_title("moon", fontsize=9)
plot_histogram(moon, axes[1], "histogram")
plt.tight_layout()
plt.show()

### TODO 4: Histogram equalization, by hand

Do not call a library function here — implement the textbook recipe:

1. histogram `h[i]` of the 8-bit image;
2. normalized cumulative distribution `cdf[i] = sum(h[0..i]) / n_pixels`;
3. look-up table `lut[i] = round(255 * cdf[i])`;
4. output image `out = lut[img]` (NumPy fancy indexing applies the LUT to every pixel).

The equalized histogram should be spread over the full 0–255 range.

In [ ]:
def equalize_manual(img):
    # TODO 4: histogram -> normalized CDF -> LUT -> apply LUT
    hist = None
    cdf = None
    lut = None
    return None


moon_eq = equalize_manual(moon)

fig, axes = plt.subplots(2, 2, figsize=(9, 6))
axes[0, 0].imshow(moon, vmin=0, vmax=255); axes[0, 0].axis("off"); axes[0, 0].set_title("before", fontsize=9)
plot_histogram(moon, axes[0, 1], "low contrast histogram")
axes[1, 0].imshow(moon_eq, vmin=0, vmax=255); axes[1, 0].axis("off"); axes[1, 0].set_title("after", fontsize=9)
plot_histogram(moon_eq, axes[1, 1], "equalized histogram")
plt.tight_layout()
plt.show()

print("std before / after :", f"{moon.std():.1f} / {moon_eq.std():.1f}")
assert moon_eq.std() > moon.std()
p_before = np.percentile(moon, [5, 95])
p_after = np.percentile(moon_eq, [5, 95])
print(f"5-95 percentile spread : {np.ptp(p_before):.0f} -> {np.ptp(p_after):.0f}")
assert np.ptp(p_after) > np.ptp(p_before)

## Part 5: Spatial filtering & convolution

A **neighborhood operation**: a kernel slides over the image, the output pixel is the
weighted sum of the neighborhood. This is the operation a CNN later *learns* instead of
having an engineer design it.

### TODO 5: Write convolution from scratch

Implement 2D convolution with zero padding and `"same"` output size.

Two details that separate convolution from correlation, and that cost people marks in
the exam:

* the kernel must be **flipped** in both directions (`k[::-1, ::-1]`);
* the output has the same shape as the input, so pad the input by `kh//2`, `kw//2`.

Your result is checked against `scipy.ndimage.convolve`.

*(A double Python loop over every pixel takes about a second for 512x512 — which is
exactly why real libraries implement this in C, and why CNNs run on GPUs.)*

In [ ]:
def convolve2d(img, kernel):
    img = img.astype(np.float64)
    kernel = np.asarray(kernel, dtype=np.float64)
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2

    # TODO 5a: flip the kernel in both directions
    k = None

    # TODO 5b: zero-pad the image by (ph, pw) on each side
    padded = None

    out = np.zeros_like(img)
    # TODO 5c: for every output pixel, sum the elementwise product of the
    #           kh x kw window of `padded` with `k`
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            out[i, j] = None
    return out


test = np.random.rand(30, 40)
kern = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]], dtype=float)
assert np.allclose(convolve2d(test, kern), ndi.convolve(test, kern, mode="constant")), "mismatch!"
print("convolve2d matches scipy.ndimage.convolve")

### TODO 6: Smoothing: which filter for which noise?

Build a 3x3 **box (arithmetic mean)** kernel and apply it with your `convolve2d` to both
noisy images. Then apply a 3x3 **median** filter (`ndi.median_filter`) to both.

Fill in the PSNR table and draw the conclusion: mean and median are *not*
interchangeable — one of them is dramatically better on impulse noise.

In [ ]:
def psnr(a, b):
    # peak signal to noise ratio for images in [0, 1]; higher is better
    mse = np.mean((a - b) ** 2)
    return 10 * np.log10(1.0 / mse)


# TODO 6a: 3x3 arithmetic mean (box) kernel
box = None

# TODO 6b: filter both noisy images with the box kernel (use YOUR convolve2d)
mean_g = None
mean_sp = None

# TODO 6c: filter both noisy images with a 3x3 median filter
med_g = None
med_sp = None

show(noisy_gauss, mean_g, med_g, titles=["gaussian noise", "3x3 mean", "3x3 median"], vmin=0, vmax=1)
show(noisy_sp, mean_sp, med_sp, titles=["salt & pepper", "3x3 mean", "3x3 median"], vmin=0, vmax=1)

print(f"{'':16s}{'noisy':>8s}{'mean':>8s}{'median':>8s}")
print(f"{'gaussian':16s}{psnr(cam_f, noisy_gauss):8.2f}{psnr(cam_f, mean_g):8.2f}{psnr(cam_f, med_g):8.2f}")
print(f"{'salt & pepper':16s}{psnr(cam_f, noisy_sp):8.2f}{psnr(cam_f, mean_sp):8.2f}{psnr(cam_f, med_sp):8.2f}")

assert psnr(cam_f, med_sp) > psnr(cam_f, mean_sp)

**Question:** the median filter is *not* a convolution. Why can it not be written as a
kernel of weights, and why does that non-linearity help against salt-and-pepper noise?

*Your answer:*



### TODO 7: Sharpening: Laplacian and unsharp masking

1. Use the Laplacian kernel from the lecture (center `8`, all neighbors `-1`).
   Because it is a high-pass filter its response is near zero on flat regions and large
   at edges. Add the response back onto the original: `sharp = f + c * laplacian(f)`.
2. Implement **unsharp masking**: `sharp = f + amount * (f - blurred)`, where `blurred`
   comes from `filters.gaussian(f, sigma=...)`.

Clip both results to [0, 1].

In [ ]:
# TODO 7a: the Laplacian kernel from the lecture slide
laplacian_kernel = None

# TODO 7b: convolve, then add the response back onto the original (weight 0.3) and clip
lap_response = None
sharp_lap = None

# TODO 7c: unsharp masking with sigma=2 and amount=1.5
blurred = None
mask = None
sharp_unsharp = None

show(cam_f, lap_response, sharp_lap, sharp_unsharp,
     titles=["original", "Laplacian response", "f + 0.3*Laplacian", "unsharp mask"],
     size=2.9)
assert sharp_lap.max() <= 1.0 and sharp_lap.min() >= 0.0

## Part 6: Edge detection

An edge is a **discontinuity**: a place where intensity changes sharply. We look for
large gradient magnitudes.

### TODO 8: Sobel gradients by hand

The two Sobel masks estimate the derivative in x (vertical changes, i.e. horizontal
edges) and in y:

```
Sx = [[-1, 0, 1],      Sy = [[-1, -2, -1],
      [-2, 0, 2],            [ 0,  0,  0],
      [-1, 0, 1]]            [ 1,  2,  1]]
```

Compute `gx`, `gy`, the gradient magnitude `sqrt(gx**2 + gy**2)`, and a binary edge map
by thresholding the magnitude at its 90th percentile.

In [ ]:
# TODO 8a: the two Sobel kernels
Sx = None
Sy = None

# TODO 8b: gradients, magnitude, and a boolean edge map at the 90th percentile
gx = None
gy = None
magnitude = None
edges_thresh = None

show(np.abs(gx), np.abs(gy), magnitude, edges_thresh,
     titles=["|gx|", "|gy|", "gradient magnitude", "thresholded (top 10%)"], size=2.9)
assert edges_thresh.dtype == bool

### Canny: the gold standard

`feature.canny` adds Gaussian smoothing, non-maximum suppression and hysteresis
thresholding on top of the gradient you just computed by hand. Compare its output at
`sigma = 1, 2, 4` against your plain threshold — note the thin, single-pixel-wide,
connected contours.

*(Worked example — read the code and run it.)*

In [ ]:
sigmas = [1, 2, 4]
cannies = [feature.canny(cam_f, sigma=s) for s in sigmas]

show(edges_thresh, *cannies,
     titles=["plain gradient threshold"] + [f"Canny sigma={s}" for s in sigmas], size=2.9)

for s, e in zip(sigmas, cannies):
    print(f"sigma={s}: {e.sum():6d} edge pixels")
assert cannies[0].sum() > cannies[-1].sum()

**Think about it:** larger `sigma` gives fewer, cleaner edges but poorer localization.
Which of Canny's three optimization criteria (low error rate, good localization, single
response) does raising `sigma` trade away?

## Part 7: Segmentation & classification

Segmentation partitions the image into regions, using **discontinuity** (edges) or
**similarity** (shared color/texture/intensity statistics).

### Otsu thresholding

`data.coins()` is a high-contrast scene: bright coins on a darker background — the case
where global thresholding is supposed to work. `filters.threshold_otsu` picks the
cut-off `T` automatically by maximizing the separation between the two intensity
classes. The red line in the histogram shows where it lands.

*(Worked example — read the code and run it.)*

In [ ]:
coins = data.coins()

T = filters.threshold_otsu(coins)
binary = coins > T

print("Otsu threshold T =", T)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
axes[0].imshow(coins); axes[0].axis("off"); axes[0].set_title("coins", fontsize=9)
plot_histogram(coins, axes[1], "histogram + threshold")
axes[1].axvline(T, color="red", lw=2)
axes[2].imshow(binary); axes[2].axis("off"); axes[2].set_title(f"binary (T={T})", fontsize=9)
plt.tight_layout()
plt.show()

assert binary.dtype == bool

### TODO 9: From pixels to *information*: count the coins

The raw binary mask is not usable yet: the coins have holes, and the background has
specks. Clean it up, then label the connected components.

First write your own **small-object filter** — it is three lines of NumPy and it reuses
the look-up-table trick from TODO 4:

1. `labels, n = ndi.label(mask)` numbers the connected components (0 = background);
2. `np.bincount(labels.ravel())` gives the pixel count of every label at once;
3. build a boolean array `keep` over the labels (force `keep[0] = False`, the background
   is never an object) and apply it as a LUT: `keep[labels]`.

Then run the pipeline: fill holes → remove small objects → label → count.
There are **24** coins in the image.

In [ ]:
def remove_small(mask, min_size):
    # TODO 9a: label -> component sizes -> boolean keep-LUT -> apply to labels
    labels, n = None, None
    sizes = None
    keep = None
    return None


# TODO 9b: fill the holes inside the coins
filled = None

# TODO 9c: drop every component smaller than 200 pixels
cleaned = None

# TODO 9d: label the connected components and count them
labels, n_objects = None, None
print("objects found:", n_objects)

show(binary, filled, cleaned, color.label2rgb(labels, bg_label=0),
     titles=["raw binary", "holes filled", "small objects removed", f"{n_objects} labelled regions"],
     size=2.9)

areas = sorted(int(r.area) for r in measure.regionprops(labels))
print("region areas:", areas)
print("largest region:", areas[-1], "px  vs  median:", int(np.median(areas)), "px")

You should get **24** — the right answer. But look at the region areas: one region is
several times bigger than a coin. Zoom into the labelled image at the top of the frame.

The background there is brighter than the global threshold `T`, so a strip of background
passed as foreground and swallowed a coin. The count is right *by accident*: one merged
region happens to stand in for one coin.

This is the built-in weakness of **global** thresholding — it assumes uniform
illumination. Part 8 attacks exactly this.

*Note:* the input to the whole notebook was an image; the output of this cell is a
**number**. That is the step from Image Processing to Computer Vision.

### K-means color segmentation

This reproduces the K-means slide: each pixel becomes a 3D feature vector `(R, G, B)`,
K-means finds the *modes* of that distribution, and every pixel is replaced by its
cluster center. Segmentation by **similarity** rather than by discontinuity — and note
that spatial position is never used, so a cluster can be scattered all over the image.

*(Worked example — read the code and run it.)*

In [ ]:
from sklearn.cluster import KMeans


def kmeans_segment(img, K):
    h, w, _ = img.shape
    X = img.reshape(-1, 3)
    km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(X)
    return km.cluster_centers_[km.labels_].reshape(h, w, 3)


kres = [kmeans_segment(chel, K) for K in (2, 3, 10)]
show(*kres, chel, titles=["K=2", "K=3", "K=10", "original"], size=2.9)

assert len(np.unique(kres[0].reshape(-1, 3), axis=0)) == 2

## Part 8: Final challenge: the full pipeline

Everything so far, in one function. You are given a **degraded** version of the coins
image: salt-and-pepper noise plus a strong brightness gradient across the frame
(uneven illumination — the first item on the *Challenges in Computer Vision* slide).

Global Otsu thresholding **fails** on this image. Show that first, then fix it.

Your `count_coins(img)` should follow the pipeline from the lecture:

```
Acquisition -> Enhancement/Restoration -> Segmentation -> Morphology -> Representation
```

Suggested recipe:

1. **Denoise** — which filter did Part 5 recommend for salt-and-pepper?
2. **Correct the illumination** — `filters.threshold_local(img, block_size=…)` computes
   a *local* threshold per pixel instead of one global `T`; compare the image against it.
   (Alternative: estimate the background with a very large Gaussian blur and subtract it.)
3. **Clean up** — fill holes, remove small objects.
4. **Count** — label the components and return the number.

Target: **24**.

In [ ]:
noisy_coins = util.random_noise(util.img_as_float(coins), mode="s&p", amount=0.05, rng=1)
rows, cols = noisy_coins.shape
gradient = np.linspace(0, 0.55, cols)[None, :].repeat(rows, axis=0)
degraded = np.clip(noisy_coins + gradient, 0, 1)

T_global = filters.threshold_otsu(degraded)
naive = degraded > T_global
naive_labels, naive_n = measure.label(
    remove_small(ndi.binary_fill_holes(naive), 300), return_num=True)

show(degraded, naive, color.label2rgb(naive_labels, bg_label=0),
     titles=["degraded input", f"global Otsu (T={T_global:.2f})", f"naive count = {naive_n}"],
     size=3.2)
print("Naive global thresholding finds", naive_n, "coins instead of 24.")

In [ ]:
def count_coins(img, block_size=101, offset=-0.05, min_size=300, return_mask=False):
    # TODO 10: implement the pipeline
    # 1. denoise (which filter did Part 5 recommend for salt & pepper?)
    den = None
    # 2. local (adaptive) thresholding -> boolean mask
    #    filters.threshold_local(den, block_size=..., offset=...) returns an ARRAY
    #    of thresholds, one per pixel - compare the image against it
    mask = None
    # 3. morphological clean-up: fill holes, then reuse your remove_small()
    #    if coins end up merged, morphology.opening(mask, morphology.disk(3))
    #    can break thin bridges between them
    # 4. label and count
    labels, n = None, None
    if return_mask:
        return n, labels
    return n


n, labels = count_coins(degraded, return_mask=True)
areas = sorted(int(r.area) for r in measure.regionprops(labels))
show(degraded, labels > 0, color.label2rgb(labels, bg_label=0),
     titles=["degraded input", "cleaned mask", f"count = {n}"], size=3.2)
print("coins found:", n)
print("areas:", areas)
assert n == 24, f"got {n}, expected 24 - tune block_size / offset / min_size"

## Wrap-up

Answer briefly, in your own words:

1. Which cells in this notebook were **Image Processing** (image in → image out) and
   which were **Computer Vision** (image in → information out)? Name the cell where the
   transition happens.
2. Part 5 asked you to *design* a kernel (Laplacian, Sobel, box) by hand. What does a
   CNN do differently, and what does it need in exchange?
3. Name one place in Part 8 where a hand-tuned parameter would break if the lighting or
   the camera changed. How would you make that step more robust?

*Your answers:*

